In [25]:
!pip install pygam

In [26]:
import pandas as pd
import numpy as np
from pygam import LinearGAM, s, te, f
from libpysal.weights import KNN
from esda.moran import Moran
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

df = pd.read_csv('/kaggle/input/datasets/shauryapandey19/delhi-spatiotemporal-pm25/delhi_10station_pm25_spatiotemporal_dataset_model_ready_v5.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.head())
print(df.shape)

                  timestamp  station_id    station_name operating_agency  \
0 2025-02-18 16:00:00+00:00       10484  aurobindo_marg      DPCC / CPCB   
1 2025-02-18 16:00:00+00:00        5570       aya_nagar       IMD / CPCB   
2 2025-02-18 16:00:00+00:00        8472          bawana      DPCC / CPCB   
3 2025-02-18 16:00:00+00:00        5626             dtu             CPCB   
4 2025-02-18 16:00:00+00:00        5622            nsut             CPCB   

                                   region_type  \
0  Major Traffic Corridor / Institutional Belt   
1         Semi-Urban / Border Residential Belt   
2               Industrial Area / Outer Suburb   
3     Institutional / Educational Complex Zone   
4         Institutional / Suburban Campus Zone   

                                    zone_description  latitude  longitude  \
0  South Delhi / Sri Aurobindo Marg arterial road...  28.53130  77.200100   
1  South Delhi / Ridge & Border Area near Gurugra...  28.47270  77.126500   
2  North We

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116874 entries, 0 to 116873
Data columns (total 38 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   timestamp                116874 non-null  datetime64[ns, UTC]
 1   station_id               116874 non-null  int64              
 2   station_name             116874 non-null  object             
 3   operating_agency         116874 non-null  object             
 4   region_type              116874 non-null  object             
 5   zone_description         116874 non-null  object             
 6   latitude                 116874 non-null  float64            
 7   longitude                116874 non-null  float64            
 8   timestamp_ist            116874 non-null  object             
 9   temperature_C            116874 non-null  float64            
 10  relative_humidity_pct    116874 non-null  int64              
 11  wind_speed_m_

### Builds PM2.5 lag features (1h, 3h, 6h, 24h) per station using timestamp-exact lookups, so each station's history stays isolated from the others.

In [28]:
# store the finished lag columns in a list of small tables
# one small table per station, then glue them all together at the end
all_station_tables = []

# Get the list of unique station names
all_stations = df['station_name'].unique()

# Go through each station one at a time
for station in all_stations:

    # Get only the rows belonging to this one station
    station_data = df[df['station_name'] == station].copy()

    # Set the timestamp as the index
    # This lets us look up "the row at exactly this timestamp" very easily
    station_data = station_data.set_index('timestamp')

    # Sort by time just to be safe
    station_data = station_data.sort_index()

    # Make a separate small series that only holds the pm25 column
    # indexed by timestamp, since that is all we need for lag lookups
    pm25_by_time = station_data['pm25']

    # Create an empty list to hold the 1-hour lag values for every row
    lag_1h_values = []

    # for 3-hour lag values for every row
    lag_3h_values = []

    # for 6-hour lag values for every row
    lag_6h_values = []

    # for 24-hour lag values 
    lag_24h_values = []

    # Go through every timestamp in this station's data, one at a time
    for current_time in station_data.index:

        # Work out what timestamp was exactly 1 hour earlier
        time_1h_ago = current_time - pd.Timedelta(hours=1)

        # Check if that exact timestamp exists in this station's data
        if time_1h_ago in pm25_by_time.index:
            # If it exists, grab the pm25 value from that exact time
            value_1h = pm25_by_time.loc[time_1h_ago]
        else:
            # If it does not exist, we have no choice but to leave it empty
            value_1h = np.nan

        # Add this value to the list
        lag_1h_values.append(value_1h)

        # Work out what timestamp was exactly 3 hours earlier
        time_3h_ago = current_time - pd.Timedelta(hours=3)

        # Check if that exact timestamp exists
        if time_3h_ago in pm25_by_time.index:
            value_3h = pm25_by_time.loc[time_3h_ago]
        else:
            value_3h = np.nan

        # Add this value to_the list
        lag_3h_values.append(value_3h)

        # Work out what timestamp was exactly 6 hours earlier
        time_6h_ago = current_time - pd.Timedelta(hours=6)

        # Check if that exact timestamp exists
        if time_6h_ago in pm25_by_time.index:
            value_6h = pm25_by_time.loc[time_6h_ago]
        else:
            value_6h = np.nan

        # Add this value to the list
        lag_6h_values.append(value_6h)

        # Work out what timestamp was exactly 24 hours earlier
        time_24h_ago = current_time - pd.Timedelta(hours=24)

        
        if time_24h_ago in pm25_by_time.index:
            value_24h = pm25_by_time.loc[time_24h_ago]
        else:
            value_24h = np.nan

        
        lag_24h_values.append(value_24h)

    # Attach all four lag lists as new columns on this station's table
    station_data['pm25_lag1'] = lag_1h_values
    station_data['pm25_lag3'] = lag_3h_values
    station_data['pm25_lag6'] = lag_6h_values
    station_data['pm25_lag24'] = lag_24h_values

    # Turn the timestamp back into a normal column instead of an index
    station_data = station_data.reset_index()

    # Save this station's finished table into our list
    all_station_tables.append(station_data)

# Glue all the station tables back together into one big table again
df = pd.concat(all_station_tables, ignore_index=True)

# Show how many values are missing in each lag column, just to check
print(df[['pm25_lag1', 'pm25_lag3', 'pm25_lag6', 'pm25_lag24']].isna().sum())

pm25_lag1      458
pm25_lag3     1283
pm25_lag6     2249
pm25_lag24    4509
dtype: int64


### Calculates the compass bearing from one station to another, used to determine wind alignment between neighboring stations.

In [29]:
# This function works out the compass bearing FROM one point TO another point
# Bearing means "which direction do you have to face to walk from A to B"
# It gives an answer in degrees, where 0 = North, 90 = East, 180 = South, 270 = West
def calculate_bearing(lat_from, lon_from, lat_to, lon_to):

    # Convert all four coordinates from degrees into radians
    # Radians are needed because that is what the math functions expect
    lat_from_rad = np.radians(lat_from)
    lon_from_rad = np.radians(lon_from)
    lat_to_rad = np.radians(lat_to)
    lon_to_rad = np.radians(lon_to)

    # Calculate the difference in longitude between the two points
    delta_lon = lon_to_rad - lon_from_rad

    # This is the standard bearing formula
    # It combines the latitude and longitude differences into one angle
    x = np.sin(delta_lon) * np.cos(lat_to_rad)

    y = (
        np.cos(lat_from_rad) * np.sin(lat_to_rad)
        - np.sin(lat_from_rad) * np.cos(lat_to_rad) * np.cos(delta_lon)
    )

    # atan2 turns the x and y values into a single angle, in radians
    bearing_rad = np.arctan2(x, y)

    # Convert the bearing back from radians into degrees
    bearing_deg = np.degrees(bearing_rad)

    # Bearings from atan2 can come out negative (like -30 degrees)
    # This line wraps any negative angle around to the 0-360 range
    bearing_deg = (bearing_deg + 360) % 360

    # Send the final bearing value back to whoever called this function
    return bearing_deg

### Scores how directly upwind a neighboring station is relative to the target station's current wind direction, returning a value between -1 and 1.

In [30]:
# This function checks how "upwind" a neighbor station is compared to the target
# It returns a number between -1 and 1
# 1 means the neighbor is directly upwind (wind is blowing from them to us)
# -1 means the neighbor is directly downwind (wind is blowing away from us, toward them)
def calculate_alignment(wind_dir_sin, wind_dir_cos, bearing_deg):

    # Convert the bearing from degrees into radians
    bearing_rad = np.radians(bearing_deg)

    # Work out sin and cos of the bearing angle
    bearing_sin = np.sin(bearing_rad)
    bearing_cos = np.cos(bearing_rad)

    # This is the trig identity for cos(wind_direction - bearing)
    # It tells us how closely the wind direction matches the bearing direction
    alignment = (
        wind_dir_cos * bearing_cos
        + wind_dir_sin * bearing_sin
    )

    # Send the alignment score back
    return alignment

### Converts the wind alignment score into a weight multiplier (0.1 to 1.0), giving upwind neighbors more influence than downwind ones.

In [31]:
# This function takes the alignment score (-1 to 1) and turns it into
# a weight multiplier between 0.1 and 1.0
# Upwind neighbors (alignment close to 1) get a multiplier close to 1.0
# Downwind neighbors (alignment close to -1) get a multiplier close to 0.1
# This way downwind stations still count a little, just much less
def alignment_to_multiplier(alignment):

    # Rescale alignment from the -1 to 1 range into the 0 to 1 range
    scaled = (alignment + 1) / 2

    # Now stretch that 0 to 1 range into the 0.1 to 1.0 range 
    multiplier = 0.1 + 0.9 * scaled

    return multiplier

### Defines the function that builds the neighborhood_state feature: a distance- and wind-weighted average of nearby stations' PM2.5, restricted to an allowed set of neighbor stations to prevent leakage.

In [32]:
def build_neighborhood_state(data, grouped_by_time, allowed_stations):

    neighborhood_results = []

    for _, target_row in data.iterrows():

        target_station = target_row['station_name']
        ts = target_row['timestamp']
        target_lat = target_row['latitude']
        target_lon = target_row['longitude']
        target_wind_sin = target_row['wind_dir_sin']
        target_wind_cos = target_row['wind_dir_cos']

        same_time_rows = grouped_by_time.get(ts)

        if same_time_rows is None:
            neighborhood_results.append({
                'station_name': target_station,
                'timestamp': ts,
                'neighborhood_state': np.nan
            })
            continue

        # restrict candidate neighbors to allowed_stations only
        same_time_rows = same_time_rows[
            same_time_rows['station_name'].isin(allowed_stations)
        ]
        neighbors = same_time_rows[
            same_time_rows['station_name'] != target_station
        ]
        neighbors = neighbors.dropna(subset=['pm25'])

        if len(neighbors) < 1:
            neighborhood_results.append({
                'station_name': target_station,
                'timestamp': ts,
                'neighborhood_state': np.nan
            })
            continue

        weighted_pm25_sum = 0
        weight_sum = 0

        for _, neighbor in neighbors.iterrows():

            lat1 = np.radians(target_lat)
            lon1 = np.radians(target_lon)
            lat2 = np.radians(neighbor['latitude'])
            lon2 = np.radians(neighbor['longitude'])

            dlat = lat2 - lat1
            dlon = lon2 - lon1

            a = (
                np.sin(dlat / 2) ** 2
                + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
            )
            distance = 2 * 6371 * np.arcsin(np.sqrt(a))
            if distance == 0:
                distance = 1e-6

            bearing = calculate_bearing(
                neighbor['latitude'], neighbor['longitude'],
                target_lat, target_lon
            )
            alignment = calculate_alignment(
                target_wind_sin, target_wind_cos, bearing
            )
            wind_multiplier = alignment_to_multiplier(alignment)

            distance_weight = 1 / (distance ** 2)
            final_weight = distance_weight * wind_multiplier

            weighted_pm25_sum += final_weight * neighbor['pm25']
            weight_sum += final_weight

        neighborhood_state = weighted_pm25_sum / weight_sum

        neighborhood_results.append({
            'station_name': target_station,
            'timestamp': ts,
            'neighborhood_state': neighborhood_state
        })

    return pd.DataFrame(neighborhood_results)

### Converts timestamp into a numeric hours-since-start value and assembles the core feature table used for the GAM, dropping rows with no target value.

In [33]:
# Create a new numeric column that represents time as a plain number
# GAM smooth functions need a number to work with, not a raw date
# hours since the very first timestamp in the data
earliest_time = df['timestamp'].min()

# Work out the time difference for every row, then convert it to hours
df['time_numeric'] = (df['timestamp'] - earliest_time).dt.total_seconds() / 3600

# pick out only the columns we actually use for the GAM
gam_table = df[[
    'pm25_original',
    'timestamp',
    'time_numeric',
    'station_name',
    'latitude',
    'longitude',
    'boundary_layer_height_m',
    'wind_speed_m_s',
    'wind_dir_sin',
    'wind_dir_cos',
    'pm25_lag1',
    'pm25_lag3',
    'pm25_lag6',
    'pm25_lag24',
    'pm25_was_interpolated'
]].copy()


print(gam_table.shape)

# Drop rows where the target value itself is missing
# cannot train or test on a row where we do not know the true answer
gam_table = gam_table.dropna(subset=['pm25_original'])


print(gam_table.shape)
print(gam_table.isna().sum())

(116874, 15)
(110182, 15)
pm25_original                 0
timestamp                     0
time_numeric                  0
station_name                  0
latitude                      0
longitude                     0
boundary_layer_height_m       0
wind_speed_m_s                0
wind_dir_sin                  0
wind_dir_cos                  0
pm25_lag1                     1
pm25_lag3                     3
pm25_lag6                  1201
pm25_lag24                 3770
pm25_was_interpolated         0
dtype: int64


In [34]:
required_columns = [
    'pm25_original',
    'boundary_layer_height_m',
    'wind_speed_m_s',
    'wind_dir_sin',
    'wind_dir_cos',
    'pm25_lag1',
    'pm25_lag3',
    'pm25_lag6',
    'pm25_lag24'
]

print(gam_table.shape)

# Only keep rows where every single one of these columns has a real value
# This way every row entering the GAM is on equal footing
gam_table = gam_table.dropna(subset=required_columns)

print(gam_table.shape)

print(gam_table.isna().sum())

(110182, 15)
(105529, 15)
pm25_original              0
timestamp                  0
time_numeric               0
station_name               0
latitude                   0
longitude                  0
boundary_layer_height_m    0
wind_speed_m_s             0
wind_dir_sin               0
wind_dir_cos               0
pm25_lag1                  0
pm25_lag3                  0
pm25_lag6                  0
pm25_lag24                 0
pm25_was_interpolated      0
dtype: int64


### Encodes station identity as a numeric code and defines the ordered list of feature columns and the target variable for the GAM.

In [37]:
gam_table['station_name'] = gam_table['station_name'].astype('category')


# pyGAM needs station_name turned into numeric codes, not text
# f() (factor term) will treat this as a categorical variable later
gam_table['station_code'] = gam_table['station_name'].cat.codes

# Pull out the target variable on its own
y = gam_table['pm25_original'].values

# Build the feature matrix in a fixed column order
# This order matters because we will refer to columns by position number below
feature_columns = [
    'time_numeric',
    'boundary_layer_height_m',
    'wind_speed_m_s',
    'wind_dir_sin',
    'wind_dir_cos',
    'latitude',
    'longitude',
    'pm25_lag1',
    'pm25_lag3',
    'pm25_lag6',
    'pm25_lag24',
    'neighborhood_state'   
]



### Defines the GAM's smooth and tensor terms, mapped to each feature's position in the feature matrix.

In [38]:
# 0  = time_numeric
# 1  = boundary_layer_height_m
# 2  = wind_speed_m_s
# 3  = wind_dir_sin
# 4  = wind_dir_cos
# 5  = latitude
# 6  = longitude
# 7  = pm25_lag1
# 8  = pm25_lag3
# 9  = pm25_lag6
# 10 = pm25_lag24
# 11 = neighborhood_state
 

gam_terms = (
    s(0)
    + s(1)
    + s(2)
    + te(3, 4)
    + te(5, 6)
    + s(7)
    + s(8)
    + s(9)
    + s(10)
    + s(11)   
)

### Runs leave-one-station-out validation: for each held-out station, rebuilds neighborhood_state using only the training stations, fits the GAM, and stores predictions.

In [40]:
# group all rows by timestamp once, so we can quickly look up
# "who else reported at this same hour" inside the loop below
grouped_by_time = dict(tuple(df.groupby('timestamp')))

loso_results = []

all_stations = gam_table['station_name'].unique()

# LOSO = Leave One Station Out
# each pass through this loop, one station is fully held out
for held_out_station in all_stations:

    # these are the only stations allowed to act as "neighbors" this fold
    training_stations = []
    for station in all_stations:
        if station != held_out_station:
            training_stations.append(station)

    train_rows = gam_table[gam_table['station_name'] != held_out_station].copy()
    test_rows = gam_table[gam_table['station_name'] == held_out_station].copy()

    if len(test_rows) < 1:
        continue

    # neighborhood_state gets built fresh here, only from training_stations,
    # so the held-out station can never leak into it
    train_neighborhood = build_neighborhood_state(
        train_rows, grouped_by_time, training_stations
    )
    test_neighborhood = build_neighborhood_state(
        test_rows, grouped_by_time, training_stations
    )

    # attach the fresh neighborhood_state values onto train/test rows
    train_rows = train_rows.merge(
        train_neighborhood, on=['station_name', 'timestamp'], how='left'
    )
    test_rows = test_rows.merge(
        test_neighborhood, on=['station_name', 'timestamp'], how='left'
    )

    train_rows = train_rows.dropna(subset=['neighborhood_state'])
    test_rows = test_rows.dropna(subset=['neighborhood_state'])

    if len(test_rows) < 1:
        continue

    X_train = train_rows[feature_columns].values
    y_train = train_rows['pm25_original'].values

    X_test = test_rows[feature_columns].values
    y_test = test_rows['pm25_original'].values

    gam_model = LinearGAM(gam_terms)
    gam_model.fit(X_train, y_train)

    y_pred = gam_model.predict(X_test)

    # store one row per prediction, so we can score everything after the loop
    for i in range(len(y_test)):
        loso_results.append({
            'station_name': held_out_station,
            'actual': y_test[i],
            'predicted': y_pred[i]
        })

    print(held_out_station)

aurobindo_marg
aya_nagar
bawana
dtu
nsut
punjabi_bagh
pusa
rk_puram
sonia_vihar
anand_vihar


In [41]:
loso_df = pd.DataFrame(loso_results)  # turn all saved predictions into one table

y_true = loso_df['actual']  # true pm2.5 values across every held-out station
y_pred = loso_df['predicted']  # predicted pm2.5 values across every held-out station

mae = mean_absolute_error(y_true, y_pred)  # average absolute size of the error
rmse = mean_squared_error(y_true, y_pred) ** 0.5  # penalizes large errors more heavily than MAE
r2 = r2_score(y_true, y_pred)  # how much of the variation the model explains, 1.0 is perfect
bias = (y_pred - y_true).mean()  # positive means the model tends to overpredict, negative means underpredict

print(f"Total predictions made: {len(loso_df)}")
print(f"GAM MAE:   {mae:.2f}")
print(f"GAM RMSE:  {rmse:.2f}")
print(f"GAM R2:    {r2:.3f}")
print(f"GAM Bias:  {bias:.2f}")

Total predictions made: 105507
GAM MAE:   17.58
GAM RMSE:  29.37
GAM R2:    0.878
GAM Bias:  0.58


In [42]:
per_station_results = []  # will hold one summary row per station

for station in loso_df['station_name'].unique():  # go through each station's results separately

    station_rows = loso_df[loso_df['station_name'] == station]  # only this station's predictions

    station_mae = mean_absolute_error(station_rows['actual'], station_rows['predicted'])  # this station's average error size
    station_rmse = mean_squared_error(station_rows['actual'], station_rows['predicted']) ** 0.5  # this station's penalized error
    station_r2 = r2_score(station_rows['actual'], station_rows['predicted'])  # how well the model explains this station alone
    station_bias = (station_rows['predicted'] - station_rows['actual']).mean()  # over or under prediction for this station

    per_station_results.append({  # save this station's summary row
        'station_name': station,
        'MAE': station_mae,
        'RMSE': station_rmse,
        'R2': station_r2,
        'Bias': station_bias,
        'n_test_points': len(station_rows)
    })

per_station_df = pd.DataFrame(per_station_results)  # combine all station summaries into one table

print(per_station_df)

     station_name        MAE       RMSE        R2       Bias  n_test_points
0  aurobindo_marg  14.604860  18.467309  0.904748  13.168606          11355
1       aya_nagar  13.362233  22.848031  0.886510  -2.078754           7439
2          bawana  21.673473  31.520098  0.893799 -16.034812          11696
3             dtu  20.448020  39.622777  0.832671  12.998655           8751
4            nsut  29.292033  47.327513  0.581858  19.755479          11325
5    punjabi_bagh  14.330779  22.270089  0.935593  -7.771428          11363
6            pusa  14.616772  23.841062  0.879082   7.658455           9054
7        rk_puram  16.102821  28.287320  0.896666 -10.287563          11436
8     sonia_vihar  12.747803  21.564866  0.933561   1.796666          11585
9     anand_vihar  17.286311  25.084931  0.927789  -9.306973          11503


In [43]:
#check performance for low & high pm2.5 days
low_mask = loso_df['actual'] < 100
high_mask = loso_df['actual'] >= 100

low_mae = mean_absolute_error(loso_df[low_mask]['actual'], loso_df[low_mask]['predicted'])
low_rmse = mean_squared_error(loso_df[low_mask]['actual'], loso_df[low_mask]['predicted']) ** 0.5

high_mae = mean_absolute_error(loso_df[high_mask]['actual'], loso_df[high_mask]['predicted'])
high_rmse = mean_squared_error(loso_df[high_mask]['actual'], loso_df[high_mask]['predicted']) ** 0.5

print(f"Low PM2.5 (<100)  MAE: {low_mae:.2f}  RMSE: {low_rmse:.2f}  n={low_mask.sum()}")
print(f"High PM2.5 (>=100) MAE: {high_mae:.2f}  RMSE: {high_rmse:.2f}  n={high_mask.sum()}")

Low PM2.5 (<100)  MAE: 13.52  RMSE: 19.31  n=77800
High PM2.5 (>=100) MAE: 29.01  RMSE: 47.32  n=27707


# =========================================
#             MIXED-EFFECTS
# =========================================

### Per-station residual bias
Average leftover error (predicted − actual) per station from the LOSO results. This is the "baseline offset" a random intercept would estimate.

In [44]:
loso_df['residual'] = loso_df['predicted'] - loso_df['actual']

station_residuals = loso_df.groupby('station_name')['residual'].agg(['mean', 'std', 'count'])
station_residuals.columns = ['mean_bias', 'residual_std', 'n']

print(station_residuals)

                mean_bias  residual_std      n
station_name                                  
anand_vihar     -9.306973     23.295519  11503
aurobindo_marg  13.168606     12.947745  11355
aya_nagar       -2.078754     22.754799   7439
bawana         -16.034812     27.137877  11696
dtu             12.998655     37.432066   8751
nsut            19.755479     43.009045  11325
punjabi_bagh    -7.771428     20.871035  11363
pusa             7.658455     22.578765   9054
rk_puram       -10.287563     26.351456  11436
sonia_vihar      1.796666     21.490819  11585


### One-sample t-test per station, checking if the average residual is significantly different from zero (vs. just noise).

In [45]:
from scipy import stats

t_test_results = []

for station in loso_df['station_name'].unique():
    station_resid = loso_df[loso_df['station_name'] == station]['residual']

    # one-sample t-test: is this station's average residual meaningfully different from 0?
    t_stat, p_value = stats.ttest_1samp(station_resid, 0)

    t_test_results.append({
        'station_name': station,
        'mean_bias': station_resid.mean(),
        't_stat': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05  # small p-value = bias is unlikely to be random chance
    })

t_test_df = pd.DataFrame(t_test_results)
print(t_test_df)

     station_name  mean_bias      t_stat        p_value  significant
0  aurobindo_marg  13.168606  108.377525   0.000000e+00         True
1       aya_nagar  -2.078754   -7.879295   3.760525e-15         True
2          bawana -16.034812  -63.900858   0.000000e+00         True
3             dtu  12.998655   32.485044  1.155207e-218         True
4            nsut  19.755479   48.881764   0.000000e+00         True
5    punjabi_bagh  -7.771428  -39.692032   0.000000e+00         True
6            pusa   7.658455   32.274625  2.080903e-216         True
7        rk_puram -10.287563  -41.748894   0.000000e+00         True
8     sonia_vihar   1.796666    8.998340   2.647284e-19         True
9     anand_vihar  -9.306973  -42.849093   0.000000e+00         True


### Variance decomposition (ICC)
Splits total residual variance into between-station (location effect) and within-station (time-varying conditions/noise), and reports what share is due to station identity.

In [46]:
station_means = loso_df.groupby('station_name')['residual'].mean()

between_station_var = station_means.var(ddof=1)
within_station_var = loso_df.groupby('station_name')['residual'].var().mean()

icc = between_station_var / (between_station_var + within_station_var)

print(f"Between-station variance: {between_station_var:.3f}")
print(f"Within-station variance:  {within_station_var:.3f}")
print(f"ICC (share of variance explained by station identity): {icc:.3f}")

Between-station variance: 144.336
Within-station variance:  731.716
ICC (share of variance explained by station identity): 0.165


## Moran's I on station residual bias
Tests whether stations that are geographically close also have similar leftover bias — i.e. whether spatial structure remains after `te(lat,lon)` in the GAM.

In [47]:
station_coords = pd.DataFrame({
    'station_name': ['anand_vihar', 'aurobindo_marg', 'aya_nagar', 'bawana', 'dtu',
                      'nsut', 'punjabi_bagh', 'pusa', 'rk_puram', 'sonia_vihar'],
    'latitude': [28.64690, 28.53130, 28.47270, 28.77620, 28.75005,
                 28.60900, 28.67400, 28.63960, 28.56420, 28.72500],
    'longitude': [77.316000, 77.200100, 77.126500, 77.051100, 77.111261,
                  77.032500, 77.131000, 77.146200, 77.177000, 77.249000]
})

In [48]:


station_coords = station_coords.merge(
    station_residuals[['mean_bias']],
    on='station_name',
    how='left'
)

print(station_coords)

coords = station_coords[['longitude', 'latitude']].values

# each station's bias "sees" its k nearest neighbors when checking spatial pattern
w = KNN.from_array(coords, k=3)
w.transform = 'r'

# align bias values to the same station order as coords
bias_values = station_coords['mean_bias'].values

moran = Moran(bias_values, w)

print(f"Moran's I: {moran.I:.3f}")
print(f"p-value: {moran.p_sim:.3f}")

     station_name  latitude  longitude  mean_bias
0     anand_vihar  28.64690  77.316000  -9.306973
1  aurobindo_marg  28.53130  77.200100  13.168606
2       aya_nagar  28.47270  77.126500  -2.078754
3          bawana  28.77620  77.051100 -16.034812
4             dtu  28.75005  77.111261  12.998655
5            nsut  28.60900  77.032500  19.755479
6    punjabi_bagh  28.67400  77.131000  -7.771428
7            pusa  28.63960  77.146200   7.658455
8        rk_puram  28.56420  77.177000 -10.287563
9     sonia_vihar  28.72500  77.249000   1.796666
Moran's I: -0.357
p-value: 0.075
